# Chapter 4 · MCP fundamentals

The tools of Northside Veterinary Clinic, first inside the agent's program and then published on three MCP servers: the clinic's, billing's and the pharmacy's, which is a separate program we talk to through stdio. We look at the JSON-RPC messages of a conversation between client and server, measure the agent with sixteen tasks in four variants, and measure what the protocol's format costs in tokens.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the model and the clinic

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0  # the versions the book was checked with

In [ ]:
"""Chapter 4: MCP fundamentals. The clinic's tools, first inside the program and then
published on three MCP servers that the agent discovers."""
import asyncio
import copy
import json
import re
import time
import subprocess
import sys
import unicodedata
import urllib.request
from contextlib import AsyncExitStack
from datetime import date
from pathlib import Path
from typing import Annotated

import torch
from mcp import Client, StdioServerParameters
from mcp.server import MCPServer
from pydantic import Field
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
MODEL = None                               # None: choose according to the hardware
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/en/"
TODAY = date(2026, 10, 20)                 # the date of the clinic's data


def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; if not, on the CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def download(name):
    """A JSON file from the data folder; in Colab, it is downloaded."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return json.loads(path.read_text(encoding="utf-8"))

In [ ]:
tok, model = load(MODEL)

In [ ]:
def reset():
    """Puts the clinic back as it was: the original schedule and no notices."""
    global APPOINTMENTS, NOTICES
    APPOINTMENTS, NOTICES = copy.deepcopy(APPOINTMENTS0), []


def strip_accents(text):
    """Lowercase and without accents, to compare names however they are written."""
    return "".join(c for c in unicodedata.normalize("NFD", str(text).lower())
                   if unicodedata.category(c) != "Mn").strip()


def book_appointment(pet, vet, date, time, reason):
    """Books a free slot in the schedule; returns the appointment or the problem."""
    if pet not in PETS:
        return {"error": f"There is no pet {pet}."}
    if vet not in VETS:
        return {"error": f"There is no vet {vet}."}
    if any(a["date"] == date and a["time"] == time and a["vet"] == vet
           and a["status"] != "canceled" for a in APPOINTMENTS):
        return {"error": f"{VETS[vet]} already has an appointment on {date} at {time}."}
    appointment = {"id": f"AP-{len(APPOINTMENTS0) + len(APPOINTMENTS) + 1:04d}",
                   "date": date, "time": time, "vet": vet, "pet": pet,
                   "reason": reason, "status": "scheduled"}
    APPOINTMENTS.append(appointment)
    return appointment


def cancel_appointment(appointment):
    """Cancels a scheduled appointment."""
    for a in APPOINTMENTS:
        if a["id"] == appointment:
            if a["status"] != "scheduled":
                return {"error": f"Appointment {appointment} is \"{a['status']}\"."}
            a["status"] = "canceled"
            return {"canceled": appointment, "date": a["date"], "time": a["time"]}
    return {"error": f"There is no appointment {appointment}."}


def notify(owner, text):
    """Sends a notice to the owner (here it is stored in a list)."""
    if owner not in OWNERS:
        return {"error": f"There is no client {owner}."}
    NOTICES.append({"owner": owner, "text": text})
    return {"sent": True, "to": OWNERS[owner]["name"],
            "via": OWNERS[owner]["notify_by"]}


def age(birth_date):
    """Full years on the date of the data."""
    b = date.fromisoformat(birth_date)
    return TODAY.year - b.year - ((TODAY.month, TODAY.day) < (b.month, b.day))


def pet_summary(pid):
    """The essentials of a pet, without the full history."""
    p = PETS[pid]
    return {"id": pid, "name": p["name"], "species": p["species"], "breed": p["breed"],
            "age_years": age(p["birth_date"]), "current_weight_kg": p["weight_kg"],
            "allergies": p["allergies"] or "none known",
            "owner": {"id": p["owner"], "name": OWNERS[p["owner"]]["name"]}}


def vaccines_with_status(pid):
    """Each vaccine with its next date and whether it is already overdue."""
    return [{"vaccine": v["vaccine"], "next": v["next"],
             "status": "overdue" if v["next"] < TODAY.isoformat() else "up to date"}
            for v in VACCINES[pid]]


def client_record_3(client_id):
    """A client's record with their pets and unpaid invoices (chapter 1, toolset 3)."""
    if client_id not in OWNERS:
        return {"error": f"There is no client {client_id}. Look them up by name with "
                         "search(table='owners', field='name', value=...)."}
    unpaid = [i for i in INVOICES if i["owner"] == client_id and not i["paid"]]
    return {"id": client_id, "name": OWNERS[client_id]["name"],
            "notify_by": OWNERS[client_id]["notify_by"],
            "pets": [{"id": p, "name": d["name"], "species": d["species"]}
                     for p, d in PETS.items() if d["owner"] == client_id],
            "unpaid_invoices": [{"id": i["id"], "date": i["date"], "total": i["total"]}
                                for i in unpaid],
            "total_unpaid": round(sum(i["total"] for i in unpaid), 2)}


def schedule_3(date):
    """A day's appointments and each vet's free times (chapter 1, toolset 3)."""
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(date)):
        return {"error": f"The date \"{date}\" is not valid; use YYYY-MM-DD."}
    appointments = [a for a in APPOINTMENTS
                    if a["date"] == date and a["status"] != "canceled"]
    return {"date": date,
            "appointments": [f"{a['id']} {a['time']} {a['vet']} "
                             f"{PETS[a['pet']]['name']} ({a['pet']})"
                             for a in appointments],
            "vets": VETS,
            "free_times": {v: [t for t in TIMES if not any(
                a["time"] == t and a["vet"] == v for a in appointments)]
                for v in VETS}}


def locate(pet, owner=""):
    """The pet's id from its name (and the owner's, if there are several)."""
    ids = [p for p, d in PETS.items() if strip_accents(pet) == strip_accents(d["name"])
           and strip_accents(owner) in strip_accents(OWNERS[d["owner"]]["name"])]
    if not ids:
        return None, {"error": f"There is no pet called \"{pet}\""
                               + (f" belonging to {owner}" if owner else "")
                               + " at the clinic."}
    if len(ids) > 1:
        return None, {"error": f"There are {len(ids)} pets called \"{pet}\"; give the "
                               "owner's name.",
                      "pets": [pet_summary(p) for p in ids]}
    return ids[0], None


def vet_of(name):
    """The vet's id from their name or their id."""
    for v, n in VETS.items():
        if strip_accents(name) in (strip_accents(v), strip_accents(n)) or \
                strip_accents(name) in strip_accents(n):
            return v
    return None


def find_pet(name, owner=""):
    """A pet's data, looked up by its name."""
    pid, error = locate(name, owner)
    return error or pet_summary(pid)


def pets_of(owner):
    """A client's pets, the client looked up by name."""
    ids = [c for c, x in OWNERS.items()
           if strip_accents(owner) in strip_accents(x["name"])]
    if not ids:
        return {"error": f"There is no client called \"{owner}\"."}
    return {"clients": [{"name": OWNERS[c]["name"], "pets": [
        {"name": x["name"], "species": x["species"], "breed": x["breed"]}
        for p, x in PETS.items() if x["owner"] == c]} for c in ids]}


def vaccines(pet, owner=""):
    """A pet's vaccines, with their status."""
    pid, error = locate(pet, owner)
    return error or {"pet": PETS[pid]["name"], "today": TODAY.isoformat(),
                     "vaccines": vaccines_with_status(pid)}


def history(pet, last=3, owner=""):
    """A pet's last visits."""
    pid, error = locate(pet, owner)
    if error:
        return error
    return {"pet": PETS[pid]["name"], "visits": [
        {k: v[k] for k in ("date", "reason", "treatment", "weight_kg")}
        for v in HISTORY[pid][-int(last):]]}


def dose(medicine, pet, owner=""):
    """A medicine's dose for a pet, with allergy and species warnings."""
    med = PHARMACY.get(strip_accents(medicine))
    if med is None:
        return {"error": f"We don't have \"{medicine}\". Medicines: "
                         + ", ".join(PHARMACY) + "."}
    pid, error = locate(pet, owner)
    if error:
        return error
    p = PETS[pid]
    warnings = []
    if strip_accents(medicine) in [strip_accents(a) for a in p["allergies"]]:
        warnings.append(f"{p['name']} is allergic to {medicine}: do not prescribe.")
    if p["species"] not in med["species"]:
        warnings.append(f"{medicine} is not indicated for {p['species']}s.")
    return {"medicine": strip_accents(medicine), "pet": p["name"],
            "weight_kg": p["weight_kg"], "dose_mg_per_kg": med["dose_mg_kg"],
            "dose_mg_per_dose": round(med["dose_mg_kg"] * p["weight_kg"], 2),
            "stock": med["stock"],
            "restock": med.get("restock", "no date") if not med["stock"] else None,
            "warnings": warnings or "none"}


def free_slots(date, vet=""):
    """A day's free times, for one vet or for all."""
    r = schedule_3(date)
    if "error" in r:
        return r
    vets = [vet_of(vet)] if vet else list(VETS)
    if vets == [None]:
        return {"error": f"There is no vet \"{vet}\". They are: "
                         + ", ".join(VETS.values()) + "."}
    return {"date": date, "free_times": {VETS[v]: r["free_times"][v] for v in vets}}


def appointments_of(pet, owner=""):
    """A pet's appointments, past and scheduled, with their ids."""
    pid, error = locate(pet, owner)
    return error or {"pet": PETS[pid]["name"], "today": TODAY.isoformat(),
                     "appointments": [
        {"id": a["id"], "date": a["date"], "time": a["time"], "reason": a["reason"],
         "vet": VETS[a["vet"]], "status": a["status"]}
        for a in APPOINTMENTS if a["pet"] == pid]}


def accounts(owner):
    """A client's unpaid invoices and the total owed."""
    ids = [c for c, x in OWNERS.items()
           if strip_accents(owner) in strip_accents(x["name"])]
    if not ids:
        return {"error": f"There is no client called \"{owner}\"."}
    return {"clients": [{k: v for k, v in client_record_3(c).items() if k != "id"}
                        for c in ids]}


def book(pet, vet, date, time, reason, owner=""):
    """Books an appointment from the pet's and the vet's names."""
    pid, error = locate(pet, owner)
    if error:
        return error
    v = vet_of(vet)
    if v is None:
        return {"error": f"There is no vet \"{vet}\". They are: "
                         + ", ".join(VETS.values()) + "."}
    r = book_appointment(pid, v, date, time, reason)
    if "error" in r:
        return r
    return {"booked": r["id"], "pet": PETS[pid]["name"], "vet": VETS[v],
            "date": date, "time": time}


def notify_owner_of(pet, text, owner=""):
    """Sends a notice to the owner of a pet."""
    pid, error = locate(pet, owner)
    return error or notify(PETS[pid]["owner"], text)


TASKS_SET = {"find_pet": find_pet, "pets_of": pets_of, "vaccines": vaccines,
             "history": history, "dose": dose, "free_slots": free_slots,
             "appointments_of": appointments_of, "accounts": accounts,
             "book_appointment": book, "cancel_appointment": cancel_appointment,
             "notify_owner": notify_owner_of}

In [ ]:
def tool(name, description, parameters):
    """A tool's schema; the parameters without a default value are required."""
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object",
                       "required": [p for p, (_, req) in parameters.items() if req],
                       "properties": {p: {"type": "string", "description": d}
                                      for p, (d, _) in parameters.items()}}}}


PET = ("Name of the pet.", True)


OWNER = ("Owner's name; only if several pets have that name.", False)


SCHEMAS_TASKS = [
    tool("find_pet", "A pet's data, by name: species, age, current weight, allergies "
         "and owner.", {"name": PET, "owner": OWNER}),
    tool("pets_of", "A client's pets, the client looked up by name.",
         {"owner": ("Owner's name.", True)}),
    tool("vaccines", "A pet's vaccines: the next date of each one and whether it is "
         "overdue.", {"pet": PET, "owner": OWNER}),
    tool("history", "A pet's last visits: date, reason, treatment and weight.",
         {"pet": PET, "last": ("How many visits (3 by default).", False),
          "owner": OWNER}),
    tool("dose", "Dose of a medicine for a pet according to its current weight, with "
         "allergy or species warnings and the stock.",
         {"medicine": ("Name of the medicine.", True), "pet": PET, "owner": OWNER}),
    tool("free_slots", "Free times in the schedule on a day, for one vet or for all.",
         {"date": ("YYYY-MM-DD.", True),
          "vet": ("Name of the vet; empty for all.", False)}),
    tool("appointments_of", "A pet's appointments (past and scheduled), with their id.",
         {"pet": PET, "owner": OWNER}),
    tool("accounts", "Unpaid invoices of a client, looked up by name, and the total "
         "owed.", {"owner": ("Owner's name.", True)}),
    tool("book_appointment", "Books an appointment at a free time.",
         {"pet": PET, "vet": ("Name of the vet.", True), "date": ("YYYY-MM-DD.", True),
          "time": ("HH:MM, on the hour or half past.", True),
          "reason": ("Reason for the appointment.", True), "owner": OWNER}),
    tool("cancel_appointment", "Cancels a scheduled appointment.",
         {"appointment": ("Id of the appointment, like AP-0026 (given by "
                          "appointments_of).", True)}),
    tool("notify_owner", "Sends a notice to a pet's owner, by email or SMS as they "
         "prefer.", {"pet": PET, "text": ("Text of the notice, in English.", True),
                     "owner": OWNER}),
]


SYSTEM = (
    "You are the assistant of the staff of Northside Veterinary Clinic, a veterinary "
    "clinic in Portland, Oregon. Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in English, in one or two sentences.")


def generate(messages, schemas, max_tokens=400):
    """One answer from the model, and the input and output tokens."""
    text = tok.apply_chat_template(messages, tools=schemas, tokenize=False,
                                   add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    new = output[0, inputs["input_ids"].shape[1]:]
    return (tok.decode(new, skip_special_tokens=True).strip(),
            inputs["input_ids"].shape[1], len(new))


def read_calls(answer):
    """The calls the model asks for, written between <tool_call> and </tool_call>."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)</tool_call>", answer, re.S):
        try:
            call = json.loads(block)
            calls.append({"name": call["name"], "arguments": call.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            calls.append({"name": None, "arguments": {}})
    return calls


def execute(call, functions):
    """Runs a call if the tool exists and the arguments fit."""
    function = functions.get(call["name"])
    if function is None:
        return {"error": f"There is no tool {call['name']}."}
    try:
        return function(**call["arguments"])
    except (TypeError, ValueError, KeyError) as e:
        return {"error": f"Invalid arguments: {e}"}


def agent(task, functions, schemas, max_steps=8, show=False):
    """Think, act, observe and repeat. It counts the tokens of the results."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    trace = []
    for step in range(1, max_steps + 1):
        answer, n_in, n_out = generate(messages, schemas)
        calls = read_calls(answer)[:1]                 # one at a time
        thought = answer.split("<tool_call>")[0].strip()
        trace.append({"step": step, "input_tokens": n_in, "output_tokens": n_out,
                      "thought": thought, "calls": []})
        if not calls:
            trace[-1]["answer"] = answer
            if show:
                print(f"[{step}] Answer: {answer}")
            return answer, trace
        if show and thought:
            print(f"[{step}] {thought}")
        messages.append({"role": "assistant", "content": thought, "tool_calls": [
            {"type": "function", "function": c} for c in calls if c["name"]]})
        for call in calls:
            result = execute(call, functions)
            text = json.dumps(result, ensure_ascii=False)
            trace[-1]["calls"].append({**call, "result": result,
                                       "result_tokens": len(tok(text)["input_ids"])})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": text})
            if show:
                args = json.dumps(call["arguments"], ensure_ascii=False)
                print(f"[{step}] {call['name']}({args})")
                print(f"     → {text[:150]}")
    return None, trace


def says(answer, *forms):
    """Does any of those forms appear in the answer?"""
    return any(re.search(f, answer or "", re.I) for f in forms)


MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August",
          "September", "October", "November", "December"]


def date_in(answer, iso):
    """Does the date appear, as 2027-08-22, 8/22/2027, August 22, 2027 or 22 August?"""
    y, m, d = iso.split("-")
    month = MONTHS[int(m) - 1][:3]
    return says(answer, iso, rf"\b0?{int(m)}/0?{int(d)}/{y}",
                rf"\b{month}[a-z]*\.? 0?{int(d)}(st|nd|rd|th)?\b",
                rf"\b0?{int(d)}(st|nd|rd|th)? (of )?{month}")


def time_in(answer, hhmm):
    """Does the time appear, as 16:30, 4:30 p.m., 4:30pm or 4:30 (4 p.m. for 16:00)?"""
    h, m = (int(x) for x in hhmm.split(":"))
    h12, right, wrong = (h - 1) % 12 + 1, "pa"[h < 12], "ap"[h < 12]
    marker, not_wrong = rf"\s*{right}\.?\s?m\b", rf"(?!\s*{wrong}\.?\s?m\b)"
    forms = [rf"\b0?{h}:{m:02d}\b", rf"\b{h12}:{m:02d}\b{not_wrong}"]
    if m == 0:
        forms += [rf"\b{h12}{marker}",
                  rf"\b{h12}\b{not_wrong}(?=\s*(,|and\b|or\b|o'clock))"]
    if (h, m) == (12, 0):
        forms.append(r"\bnoon\b")
    return says(answer, *forms)


def new_appointments():
    """The appointments created during the task."""
    return [a for a in APPOINTMENTS if a["id"] not in {x["id"] for x in APPOINTMENTS0}]


def first_free(date, vet):
    """The first free time of a vet on a day, in the original schedule."""
    taken = {a["time"] for a in APPOINTMENTS0 if a["date"] == date and a["vet"] == vet}
    return next(t for t in TIMES if t not in taken)


def owes(name):
    """What a client owes in unpaid invoices."""
    return round(sum(i["total"] for i in INVOICES if not i["paid"]
                     and OWNERS[i["owner"]]["name"] == name), 2)


def amount_in(answer, value):
    """Does the amount appear, as 136, 136.00, $136 or 136 dollars?"""
    whole = f"{value:.2f}".replace(".00", "").replace(".", r"\.")
    return says(answer, rf"\b{whole}(\.00?)?\b")


def notice_to(name, *forms):
    """Was a notice sent to that client, with all those forms in its text?"""
    return any(OWNERS[n["owner"]]["name"] == name
               and all(re.search(f, n["text"], re.I) for f in forms) for n in NOTICES)


TASKS = [
    ("rabies_vaccine", "When is Lola, Martha Fisher's dog, due for her next rabies "
     "vaccine?",
     lambda r: date_in(r, "2027-08-22")),
    ("dose", "How many milligrams of meloxicam should Max get per dose?",
     lambda r: says(r, r"2\.07", r"2\.1\b")),
    ("allergy", "They're going to prescribe amoxicillin for Simba. Is there any "
     "problem?",
     lambda r: says(r, r"allerg")),
    ("book", "Book a checkup for Toby with Dr. Paul Meyer on Thursday the 22nd, at the "
     "first free time.",
     lambda r: [(a["pet"], a["vet"], a["date"], a["time"]) for a in new_appointments()]
     == [("P-002", "V2", "2026-10-22", first_free("2026-10-22", "V2"))]),
    ("cancel", "Cancel Rocky's appointment tomorrow.",
     lambda r: [a["id"] for a in APPOINTMENTS if a["status"] == "canceled"]
     == ["AP-0060"]),
    ("overdue_notice", "Let Toby's owner know about his overdue vaccines.",
     lambda r: notice_to("Peter Adams", "rabies", "DHPP", "leptospirosis")),
    ("debt", "How much does Oscar Price owe in unpaid invoices?",
     lambda r: amount_in(r, owes("Oscar Price"))),
    ("marthas_cat", "Martha Fisher is calling about her cat. Is any of her vaccines "
     "overdue?",
     lambda r: says(r, "FVRCP")
     and says(r, "overdue|expired|past due|out of date|lapsed")),
    ("weight", "How much did Zeus weigh at his last visit, and how much has it changed "
     "since the one before?",
     lambda r: says(r, r"32\.2") and says(r, r"0\.6")
     and not says(r, r"(last|latest|most recent) visit[^.,]{0,30}(2025|31\.6)",
                  r"31\.6 kg (at|on|in|during) (his|the) (last|latest|most recent)")),
    ("last_visit", "Why did Willow come in last time, and what treatment did she get?",
     lambda r: says(r, "wound") and says(r, "amoxicillin")),
    ("free_slots", "What times is Dr. Anna Sutton free on Wednesday the 21st in the "
     "afternoon?",
     lambda r: all(time_in(r, t) for t in ("16:00", "16:30", "17:30", "18:00", "18:30"))
     and not time_in(r, "17:00")),
    ("does_not_exist", "When is Buster due for his rabies vaccine?",
     lambda r: says(r, r"\bno (pet|patient|record|animal|such|one|information)\b",
                    r"(n't|not|unable to) (exist|find|locate|appear|registered)",
                    r"\bnot (found|a patient|registered|on file|in our)",
                    r"\bthere (is|are) no\b", r"\bnone\b")
     and not says(r, r"\b20\d\d-\d\d-\d\d\b", r"\b\d\d?/\d\d?/20\d\d\b")),
]

## 2. Before: everything inside the program

A copy of the pharmacy's two functions, with their schemas written by hand.

In [ ]:
def roots(text):
    """The first seven letters of each word, without accents, to compare names."""
    return {w[:7] for w in re.findall(r"\w{4,}", strip_accents(text))}


def price(service):
    """Price in dollars of a clinic service or a medicine."""
    prices = {**PRICES, **{k: d["price"] for k, d in PHARMACY.items()}}
    shared = {k: len(roots(service) & roots(k)) for k in prices}
    best = max(shared.values())
    if best == 0:
        return {"error": f"There is no price for \"{service}\". Services: "
                         + ", ".join(PRICES) + "; and the pharmacy's medicines."}
    return {"prices_usd": {k: prices[k] for k, c in shared.items() if c == best}}


def stock(medicine):
    """Units of a medicine in the pharmacy and, if none are left, when they arrive."""
    key = next((k for k in PHARMACY if roots(k) & roots(medicine)), None)
    if key is None:
        return {"error": f"We don't have \"{medicine}\". Medicines: "
                         + ", ".join(PHARMACY) + "."}
    m = PHARMACY[key]
    return {"medicine": key, "presentation": m["presentation"], "stock": m["stock"],
            "restock": m.get("restock", "no date") if not m["stock"] else None}


FUNCTIONS = {**TASKS_SET, "price": price, "stock": stock}


SCHEMAS = SCHEMAS_TASKS + [
    tool("price", "Price in dollars of a service (consultation, vaccines, tests…) or "
         "a medicine.", {"service": ("Service or medicine.", True)}),
    tool("stock", "Units of a medicine in the pharmacy and, if none are left, when "
         "they arrive.", {"medicine": ("Name of the medicine.", True)})]


def load_data():
    """Loads the clinic's data and leaves the clinic as it was at the start."""
    global PETS, OWNERS, HISTORY, VACCINES, APPOINTMENTS0, PHARMACY, INVOICES
    global VETS, TIMES, PRICES
    PETS, OWNERS = download("pets.json"), download("owners.json")
    HISTORY, VACCINES = download("history.json"), download("vaccines.json")
    APPOINTMENTS0, PHARMACY = download("appointments.json"), download("pharmacy.json")
    INVOICES, VETS = download("invoices.json"), download("vets.json")
    PRICES = download("prices.json")
    TIMES = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    reset()

In [ ]:
load_data()
print(stock("doxycycline"))

## 3. The pharmacy server, a separate program

It is saved in a file so that it can be launched as a process.

In [ ]:
%%writefile pharmacy_server.py
"""MCP server of the clinic's pharmacy: prices and stock.

It is a separate program, as if another team had written it: it shares no code with the
agent. It talks through standard input and output (stdio), with JSON-RPC messages.
Usage: python pharmacy_server.py [data_folder]
"""
import json
import re
import sys
import unicodedata
from pathlib import Path
from typing import Annotated

from mcp.server import MCPServer
from pydantic import Field

DATA = Path(sys.argv[1] if len(sys.argv) > 1 else "data")
PHARMACY = json.loads((DATA / "pharmacy.json").read_text(encoding="utf-8"))
PRICES = json.loads((DATA / "prices.json").read_text(encoding="utf-8"))

pharmacy = MCPServer("pharmacy", instructions="Prices and stock of the clinic.")


def roots(text):
    """The first seven letters of each word, without accents, to compare names."""
    text = "".join(c for c in unicodedata.normalize("NFD", text.lower())
                   if unicodedata.category(c) != "Mn")
    return {w[:7] for w in re.findall(r"\w{4,}", text)}


@pharmacy.tool()
def price(service: Annotated[str, Field(description="Service or medicine.")]) -> dict:
    """Price in dollars of a service (consultation, vaccines, tests…) or a medicine."""
    prices = {**PRICES, **{k: d["price"] for k, d in PHARMACY.items()}}
    shared = {k: len(roots(service) & roots(k)) for k in prices}
    best = max(shared.values())
    if best == 0:
        return {"error": f"There is no price for \"{service}\". Services: "
                         + ", ".join(PRICES) + "; and the pharmacy's medicines."}
    return {"prices_usd": {k: prices[k] for k, c in shared.items() if c == best}}


@pharmacy.tool()
def stock(medicine: Annotated[str, Field(description="Name of the medicine.")]) -> dict:
    """Units of a medicine in the pharmacy and, if none are left, when they arrive."""
    key = next((k for k in PHARMACY if roots(k) & roots(medicine)), None)
    if key is None:
        return {"error": f"We don't have \"{medicine}\". Medicines: "
                         + ", ".join(PHARMACY) + "."}
    m = PHARMACY[key]
    return {"medicine": key, "presentation": m["presentation"], "stock": m["stock"],
            "restock": m.get("restock", "no date") if not m["stock"] else None}


if __name__ == "__main__":
    pharmacy.run()          # by default, stdio

## 4. What travels over the wire

The JSON-RPC messages, written by hand, one line each.

In [ ]:
# The pharmacy is another program, pharmacy_server.py; it is launched as a separate
# process, and we talk to it through its standard input and output (stdio).
PHARMACY_SERVER = Path("pharmacy_server.py")  # in the notebook, in the working folder


pharmacy = StdioServerParameters(command=sys.executable,
                                 args=[str(PHARMACY_SERVER), str(DATA)])

In [ ]:
def raw_conversation():
    """Talks to the pharmacy server by writing the JSON-RPC messages by hand, one line
    per message, and returns what is sent (>>) and what is received (<<)."""
    process = subprocess.Popen(
        [sys.executable, str(PHARMACY_SERVER), str(DATA)], stdin=subprocess.PIPE,
        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, encoding="utf-8")
    log = []

    def send(message, reply=True):
        """Writes a message as a line of JSON and, if one is due, reads the reply."""
        log.append((">>", message))
        process.stdin.write(json.dumps(message, ensure_ascii=False) + "\n")
        process.stdin.flush()
        if reply:
            log.append(("<<", json.loads(process.stdout.readline())))

    send({"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {
        "protocolVersion": "2025-06-18", "capabilities": {},
        "clientInfo": {"name": "front-desk", "version": "1.0"}}})
    send({"jsonrpc": "2.0", "method": "notifications/initialized"}, reply=False)
    send({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})
    send({"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {
        "name": "stock", "arguments": {"medicine": "doxycycline"}}})
    process.stdin.close()
    process.wait(timeout=30)
    return log

In [ ]:
for direction, message in raw_conversation():
    print(direction, json.dumps(message, ensure_ascii=False)[:400])

## 5. The clinic and billing servers

In [ ]:
Pet = Annotated[str, Field(description="Name of the pet.")]


Owner = Annotated[str, Field(description="Owner's name; only if several pets have "
                                         "that name.")]


Date = Annotated[str, Field(description="YYYY-MM-DD.")]


clinic = MCPServer("clinic", instructions="Pets, vaccines, medical history, schedule "
                   "and notices of Northside Veterinary Clinic.")


@clinic.tool(name="find_pet")
def mcp_find_pet(name: Pet, owner: Owner = "") -> dict:
    """A pet's data, by name: species, age, current weight, allergies and owner."""
    return find_pet(name, owner)


@clinic.tool(name="pets_of")
def mcp_pets_of(owner: Annotated[str, Field(description="Owner's name.")]) -> dict:
    """A client's pets, the client looked up by name."""
    return pets_of(owner)


@clinic.tool(name="vaccines")
def mcp_vaccines(pet: Pet, owner: Owner = "") -> dict:
    """A pet's vaccines: the next date of each one and whether it is overdue."""
    return vaccines(pet, owner)


@clinic.tool(name="history")
def mcp_history(pet: Pet,
                last: Annotated[int, Field(description="How many visits.")] = 3,
                owner: Owner = "") -> dict:
    """A pet's last visits: date, reason, treatment and weight."""
    return history(pet, last, owner)


@clinic.tool(name="dose")
def mcp_dose(medicine: Annotated[str, Field(description="Name of the medicine.")],
             pet: Pet, owner: Owner = "") -> dict:
    """Dose of a medicine for the current weight, warning of allergy or species."""
    return dose(medicine, pet, owner)


@clinic.tool(name="free_slots")
def mcp_free_slots(date: Date, vet: Annotated[str, Field(
        description="Name of the vet; empty for all.")] = "") -> dict:
    """Free times in the schedule on a day, for one vet or for all."""
    return free_slots(date, vet)


@clinic.tool(name="appointments_of")
def mcp_appointments_of(pet: Pet, owner: Owner = "") -> dict:
    """A pet's appointments (past and scheduled), with their id."""
    return appointments_of(pet, owner)


@clinic.tool(name="book_appointment")
def mcp_book_appointment(pet: Pet,
                         vet: Annotated[str, Field(description="Name of the vet.")],
                         date: Date,
                         time: Annotated[str, Field(
                             description="HH:MM, on the hour or half past.")],
                         reason: Annotated[str, Field(
                             description="Reason for the appointment.")],
                         owner: Owner = "") -> dict:
    """Books an appointment at a free time."""
    return book(pet, vet, date, time, reason, owner)


@clinic.tool(name="cancel_appointment")
def mcp_cancel_appointment(appointment: Annotated[str, Field(
        description="Id of the appointment, like AP-0026 (given by "
                    "appointments_of).")]) -> dict:
    """Cancels a scheduled appointment."""
    return cancel_appointment(appointment)


@clinic.tool(name="notify_owner")
def mcp_notify_owner(pet: Pet,
                     text: Annotated[str, Field(
                         description="Text of the notice, in English.")],
                     owner: Owner = "") -> dict:
    """Sends a notice to a pet's owner, by email or SMS as they prefer."""
    return notify_owner_of(pet, text, owner)


billing = MCPServer("billing", instructions="Invoices of the clinic's clients.")


@billing.tool(name="accounts")
def mcp_accounts(owner: Annotated[str, Field(description="Owner's name.")]) -> dict:
    """Unpaid invoices of a client, looked up by name, and the total owed."""
    return accounts(owner)

## 6. The host and the agent

In [ ]:
def without_titles(schema):
    """The schema without the "title" keys that Pydantic adds (the model needs none)."""
    if isinstance(schema, dict):
        return {k: without_titles(v) for k, v in schema.items() if k != "title"}
    return schema


def compact(text):
    """A result's JSON on a single line, without the indentation the server sends."""
    try:
        return json.dumps(json.loads(text), ensure_ascii=False)
    except ValueError:
        return text


class Host:
    """Connects to several MCP servers, gathers their tools and sends each call to the
    server that offers it. It knows nothing about the clinic. With compact, it removes
    the titles from the schemas and writes the results on a single line."""

    def __init__(self, servers, compact=False):
        self.servers, self.compact = servers, compact

    async def __aenter__(self):
        self.stack = AsyncExitStack()
        self.route, self.schemas = {}, []
        for server in self.servers:
            client = await self.stack.enter_async_context(Client(server))
            for t in (await client.list_tools()).tools:
                self.route[t.name] = client
                parameters = without_titles(t.input_schema) if self.compact \
                    else t.input_schema
                self.schemas.append({"type": "function", "function": {
                    "name": t.name, "description": t.description,
                    "parameters": parameters}})
        return self

    async def __aexit__(self, *error):
        await self.stack.aclose()

    async def call(self, name, arguments):
        """Calls the tool on its server and returns the text of the result."""
        client = self.route.get(name)
        if client is None:
            return json.dumps({"error": f"There is no tool {name}."},
                              ensure_ascii=False)
        r = await client.call_tool(name, arguments)
        text = "\n".join(b.text for b in r.content if b.type == "text")
        return compact(text) if self.compact else text


async def mcp_agent(task, host, max_steps=8, show=False):
    """Chapter 1's loop, with the tools of the MCP servers."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    trace = []
    for step in range(1, max_steps + 1):
        answer, n_in, n_out = generate(messages, host.schemas)
        calls = read_calls(answer)[:1]
        thought = answer.split("<tool_call>")[0].strip()
        trace.append({"step": step, "input_tokens": n_in, "thought": thought,
                      "calls": []})
        if not calls:
            if show:
                print(f"[{step}] Answer: {answer}")
            return answer, trace
        messages.append({"role": "assistant", "content": thought, "tool_calls": [
            {"type": "function", "function": c} for c in calls if c["name"]]})
        for call in calls:
            text = await host.call(call["name"], call["arguments"])
            trace[-1]["calls"].append({**call, "result": text,
                                       "result_tokens": len(tok(text)["input_ids"])})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": text})
            if show:
                args = json.dumps(call["arguments"], ensure_ascii=False)
                print(f"[{step}] {call['name']}({args})")
                print(f"     → {text[:150]}")
    return None, trace

In [ ]:
async def try_it():
    async with Host([clinic, pharmacy, billing]) as h:
        print([s["function"]["name"] for s in h.schemas])
        reset()
        await mcp_agent("How much is doxycycline, and do we have it right now?",
                        h, show=True)

await try_it()

## 7. Sixteen tasks and four variants

In [ ]:
def number_in(answer, value):
    """Does the figure appear, with one or two decimals?"""
    return says(answer, *{f"{value:.2f}".replace(".", r"\."),
                          f"{value:.1f}".replace(".", r"\.") + r"\b",
                          rf"\b{value:g}\b".replace(".", r"\.")})


PHARMACY_TASKS = [
    ("doxycycline", "How much is doxycycline, and do we have it right now?",
     lambda r: number_in(r, 14.2)
     and says(r, "out of stock", "none left", r"not (currently )?available",
              "zero units", r"(do not|don't) have (any|it)", "not in stock",
              r"no (stock|units)", r"\b(run|ran) out", r"\b0\b")),
    ("when_it_arrives", "We've run out of doxycycline. When does it arrive?",
     lambda r: date_in(r, PHARMACY["doxycycline"]["restock"])),
    ("meloxicam", "How many units of meloxicam do we have left?",
     lambda r: says(r, rf"\b{PHARMACY['meloxicam']['stock']}\b")),
    ("tobys_vaccines", "How much would it cost to give Toby all his overdue vaccines?",
     lambda r: amount_in(r, sum(PRICES[v["vaccine"] + " vaccine"]
                                for v in vaccines("Toby")["vaccines"]
                                if v["status"] == "overdue"))),
]


BANK = TASKS + PHARMACY_TASKS


def record(key, check, answer, trace, start):
    """A task's row: whether it was done right, the calls, the tokens and the time."""
    calls = [c for s in trace for c in s["calls"]]
    return {"task": key, "ok": bool(check(answer)), "answer": answer,
            "steps": len(trace), "calls": [(c["name"], c["arguments"]) for c in calls],
            "result_tokens": sum(c["result_tokens"] for c in calls),
            "input_tokens": sum(s["input_tokens"] for s in trace),
            "seconds": round(time.time() - start, 1), "trace": trace}


def solve_before(tasks=BANK):
    """Chapter 1's agent with the tools inside the program."""
    rows = []
    for key, task, check in tasks:
        reset()
        start = time.time()
        answer, trace = agent(task, FUNCTIONS, SCHEMAS)
        rows.append(record(key, check, answer, trace, start))
    return rows


async def solve_mcp(servers, compact=False, tasks=BANK):
    """The same agent, with the tools of the MCP servers."""
    rows = []
    async with Host(servers, compact) as host:
        for key, task, check in tasks:
            reset()
            start = time.time()
            answer, trace = await mcp_agent(task, host)
            rows.append(record(key, check, answer, trace, start))
        schemas = host.schemas
    return rows, schemas


def summary(rows, schemas):
    """Tasks done right, steps, calls and tokens of a variant."""
    calls = sum(len(r["calls"]) for r in rows)
    return {"ok": sum(r["ok"] for r in rows), "steps": sum(r["steps"] for r in rows),
            "calls": calls,
            "tokens_per_result": round(sum(r["result_tokens"] for r in rows)
                                       / max(calls, 1)),
            "schema_tokens": len(tok(json.dumps(schemas, ensure_ascii=False))
                                 ["input_ids"]),
            "input_tokens": sum(r["input_tokens"] for r in rows)}

In [ ]:
everything = [clinic, pharmacy, billing]
results = {}
for name, servers, compact_host in [
        ("Before: tools in the program", None, False),
        ("MCP: clinic only", [clinic], False),
        ("MCP: three servers", everything, False),
        ("MCP: three servers, compact host", everything, True)]:
    if servers is None:
        rows, schemas = solve_before(), SCHEMAS
    else:
        rows, schemas = await solve_mcp(servers, compact_host)
    results[name] = rows
    print(f"{name:42s}", summary(rows, schemas))

## Exercises

1. Explain in your own words what the host, the client and the server do at the clinic. How many clients does the agent have with the three servers?

2. Run `raw_conversation()` and find the server's capabilities in the answer to `initialize`. What do you think `listChanged` means?

3. Compare the schema of `stock` written by hand in the before with the one that arrives through `tools/list`. Which keys are left over? How many tokens does each take?

4. Make a tool of the pharmacy server return a text that is already compact, a string with `json.dumps`, instead of a dictionary. Does what reaches the client change? Is it better to fix it in the server or in the host?

5. Add to the pharmacy a tool `medicines_for(species)` that lists the medicines indicated for a species. What do you have to change in the agent so that it uses it?

6. Launch the agent with only the pharmacy server and ask it for Max's dose of meloxicam. What does it answer? Does it make anything up?

7. Measure how long a call to `stock` through stdio takes and one to `accounts` in the same program, with `time.perf_counter`, repeating a hundred times. Does the difference matter compared with what the model takes in each step?

8. What happens if two servers publish a tool with the same name? Try it by adding `price` to the billing server too. Which one does the call reach in the chapter's `Host`?

9. Follow the documentation of a desktop assistant that supports MCP servers, configure it to launch the pharmacy server, and ask it the four pharmacy questions. Compare its answers with the book's agent.

10. **Project.** Turn a tool you use in your work or studies into an MCP server, with two or three functions, and connect it to the chapter's agent next to the clinic's. Write three tasks that need it and measure the agent with and without it.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [MCP specification](https://modelcontextprotocol.io/specification) and [Python SDK](https://github.com/modelcontextprotocol/python-sdk). [Introducing the Model Context Protocol](https://www.anthropic.com/news/model-context-protocol) (Anthropic, 2024). Northside Veterinary Clinic and its data are fictional. Original code and experiments.